# 水果新鮮度 CNN 分類：Colab 執行
從下載資料、訓練、評估、Demo 到產生報告，由上往下依序執行即可。

1. 先到 **執行階段 → 變更執行階段類型 → T4 GPU**
2. 下載資料約 1~3 分鐘（3.8 GB，免 Kaggle 帳號），讀取 + 訓練約 5~10 分鐘
3. 第 4 步需要上傳幾張自己拍的蘋果／香蕉／橘子照片
4. 最後一步會下載 `fruit_results.zip`，解壓到本機 repo 後 commit 即可

In [ ]:
!nvidia-smi -L || echo "沒有 GPU，會改用 CPU 訓練（較慢）"

## 1. 取得程式與套件

In [ ]:
%cd /content
!rm -rf fruit_cnn && git clone -q https://github.com/jerry910127/-.git fruit_cnn
%cd /content/fruit_cnn
!pip install -q reportlab
# PDF 用的中文字型；裝不起來也沒關係，make_report.py 會改用內建字型
!apt-get -qq install -y fonts-wqy-zenhei > /dev/null 2>&1 && echo "中文字型 OK"

## 2. 下載資料集
Kaggle「Fruits fresh and rotten for classification」，存成 `fruit/fruit.zip`，不用解壓縮。

In [ ]:
import os, zipfile
if not os.path.exists('fruit/fruit.zip'):
    !curl -sSL -o fruit/fruit.zip https://www.kaggle.com/api/v1/datasets/download/sriramr/fruits-fresh-and-rotten-for-classification
names = zipfile.ZipFile('fruit/fruit.zip').namelist()
n_train = sum(n.startswith('dataset/train/') for n in names)
n_test = sum(n.startswith('dataset/test/') for n in names)
print(f'{os.path.getsize("fruit/fruit.zip")/1e9:.2f} GB｜train {n_train:,} 張｜test {n_test:,} 張')
assert n_train and n_test, 'zip 內容不對，請刪掉 fruit/fruit.zip 重新下載'

## 3. 訓練
按**原始照片分組**切分 train / val / test（同一張照片的旋轉、平移等變體不會跨組，避免資料洩漏）→ EDA 圖 → 訓練（最多 15 回合，5 回合沒進步就提前停止）→ 測試評估 → 所有圖表。

輸出第一行會顯示洩漏比例：原始切分約 100% → 本次 0%。

In [ ]:
!python fruit/train.py

In [ ]:
from IPython.display import Image, display
for f in ['eda_class_counts', 'eda_samples', 'learning_curves', 'confusion_matrix', 'misclassified']:
    print(f)
    display(Image(f'fruit/result/{f}.png', width=700))

### （可略過）重跑原始切分作對照
repo 已附上原始切分在 T4 上的結果（`fruit/result_original_split/`），報告會自動比較修正前後；想自己重現再執行這格（約 2 分鐘）。

In [ ]:
# !python fruit/train.py --split original --out-dir fruit/result_original_split

## 4. Demo：上傳自己拍的水果照片
可一次選多張（jpg / png）。

In [ ]:
from google.colab import files
os.makedirs('demo_photos', exist_ok=True)
for name, data in files.upload().items():
    open(f'demo_photos/{name}', 'wb').write(data)
!python fruit/predict.py demo_photos
display(Image('fruit/result/demo_predictions.png', width=700))

## 5. 產生 PDF 報告並更新 README 結果表

In [ ]:
!python fruit/make_report.py
text = open('README.md', encoding='utf-8').read()
print(text[text.index('<!-- results:start -->'):text.index('<!-- results:end -->')])

## 6. 下載結果
內含 `fruit/result/`、`fruit/output/pdf/` 與更新後的 `README.md`。解壓到本機 repo 根目錄覆蓋後 commit、push。

In [ ]:
!rm -f fruit_results.zip && zip -qr fruit_results.zip fruit/result fruit/result_original_split fruit/output README.md
files.download('fruit_results.zip')